<a href="https://colab.research.google.com/github/dhruthi22/FoKG-Mini-Project-Group-21/blob/main/FOKG_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install networkx

In [14]:
!pip install rdflib

In [15]:

# STEP 1: IMPORT LIBRARIES AND DEFINE INPUT PATHS

import os
from collections import defaultdict
import numpy as np
import pandas as pd
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import RDF, XSD

# File paths pointing to the N-Triples training and test datasets in Google Colab
TRAIN_FILE = "/content/KG-2022-train.nt.txt"
TEST_FILE = "/content/KG-2022-test.nt.txt"



# STEP 2: RDF PARSING FUNCTION
# Parses RDF N-Triples statements and reconstructs statement objects into a DataFrame

def parse_rdf_to_df(file_path, is_train=True):
    """
    Reads an RDF file in N-Triples (.nt) format and extracts fact statements.

    Parameters:
        file_path (str): Path to the .nt dataset file.
        is_train (bool): Flag indicating if the dataset is for training (contains ground truth values).

    Returns:
        pd.DataFrame: Structured DataFrame containing fact URIs, subjects, predicates, objects, and truth values.
    """
    print(f"Loading and parsing {file_path}...")
    g = Graph()
    g.parse(file_path, format="nt")

    # Group reified RDF triples by their statement ID (s_str)
    statements = defaultdict(dict)
    for s, p, o in g:
        s_str, p_str, o_str = str(s), str(p), str(o)

        if "subject" in p_str:
            statements[s_str]["subject"] = o_str
        elif "predicate" in p_str:
            statements[s_str]["predicate"] = o_str
        elif "object" in p_str:
            statements[s_str]["object"] = o_str
        elif "hasTruthValue" in p_str:
            statements[s_str]["truth"] = float(o_str)

    # Reconstruct statements into a list of dictionaries
    rows = []
    for stmt_id, data in statements.items():
        # Ensure only complete statements with subject, predicate, and object are added
        if all(k in data for k in ["subject", "predicate", "object"]):
            rows.append({
                "fact_uri": stmt_id,
                "subject": data["subject"],
                "predicate": data["predicate"],
                "object": data["object"],
                "truth": data.get("truth", None if not is_train else 1.0) # Assign None to test data
            })
    return pd.DataFrame(rows)



# STEP 3: LOAD DATASETS & PREPARE FOR FEATURE ENGINEERING

# Parse training and testing graphs into pandas DataFrames
train_df = parse_rdf_to_df(TRAIN_FILE, is_train=True)
test_df = parse_rdf_to_df(TEST_FILE, is_train=False)

# Explicitly assign NaN to test set ground truth to prevent accidental leakage during training
test_df["truth"] = np.nan

# Output parsed summary
print(f"Loaded {len(train_df)} training facts and {len(test_df)} test facts.")

Loading and parsing /content/KG-2022-train.nt.txt...
Loading and parsing /content/KG-2022-test.nt.txt...
Loaded 1234 training facts and 1342 test facts.


In [16]:

# FEATURE ENGINEERING & MODEL TRAINING PIPELINE
# Extracts global semantic features, trains Gradient Boosting, & scores test data

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score


# 1. GLOBAL FEATURE ENGINEERING
# Combine train and test datasets to compute frequency maps across all entities

combined_df = pd.concat([train_df, test_df], ignore_index=True)

print("Engineering global semantic features...")

# Extract human-readable string labels from URIs (e.g., entity names/relations)
combined_df["sub_name"] = combined_df["subject"].apply(lambda x: str(x).split("/")[-1].replace("_", " "))
combined_df["obj_name"] = combined_df["object"].apply(lambda x: str(x).split("/")[-1].replace("_", " "))
combined_df["rel_name"] = combined_df["predicate"].apply(lambda x: str(x).split("/")[-1])

# Frequency counts across combined train and test sets to detect entity popularity
combined_df["sub_freq"] = combined_df["subject"].map(combined_df["subject"].value_counts())
combined_df["obj_freq"] = combined_df["object"].map(combined_df["object"].value_counts())
combined_df["rel_freq"] = combined_df["predicate"].map(combined_df["predicate"].value_counts())

# Object-Predicate compatibility: Calculate conditional probability P(Object | Predicate)
pred_obj_counts = combined_df.groupby("predicate")["object"].value_counts().to_dict()
pred_counts = combined_df["predicate"].value_counts().to_dict()
combined_df["prob_obj_given_pred"] = combined_df.apply(
    lambda r: pred_obj_counts.get((r["predicate"], r["object"]), 0) / pred_counts.get(r["predicate"], 1), axis=1
)

# Heuristic check: Detect whether object contains numerical values/digits
combined_df["obj_is_digit"] = combined_df["obj_name"].apply(lambda x: 1 if any(char.isdigit() for char in x) else 0)

# Predicate-specific object occurrence frequency
pred_obj_freq = combined_df.groupby('predicate')['object'].transform('count')
combined_df["pred_obj_freq"] = pred_obj_freq

# Define feature columns used for model training (including pred_obj_freq)
feature_cols = [
    "sub_freq",
    "obj_freq",
    "rel_freq",
    "prob_obj_given_pred",
    "obj_is_digit",
    "pred_obj_freq"
]

# Separate combined dataset back into distinct training and test sets
final_train = combined_df[combined_df["truth"].notna()].copy()
final_test = combined_df[combined_df["truth"].isna()].copy()



# 2. MODEL TRAINING & LOCAL VALIDATION
# Train Gradient Boosting Classifier using a 80/20 train-validation split

X = final_train[feature_cols]
y = final_train["truth"]

# Create internal validation split to monitor model performance
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Gradient Boosting Classifier...")
model = GradientBoostingClassifier(
    n_estimators=200,    # Total boosting stages
    learning_rate=0.05,  # Reduced learning rate for better generalization
    max_depth=6,         # Tree depth limit to capture feature interactions
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate local validation metrics
val_preds_proba = model.predict_proba(X_val)[:, 1]
val_preds_binary = model.predict(X_val)
print(f"\nValidation Accuracy: {accuracy_score(y_val, val_preds_binary) * 100:.2f}%")
print(f"Validation ROC-AUC Score: {roc_auc_score(y_val, val_preds_proba):.4f}")


# ------------------------------------------------------------------------------
# 3. TEST SET PREDICTION
# Predict veracity scores for unlabeled test facts
# ------------------------------------------------------------------------------
print(f"\nGenerating final veracity scores for {len(final_test)} test facts...")
X_test = final_test[feature_cols]

# Store continuous probability predictions (0.0 to 1.0) in predicted_truth column
final_test["predicted_truth"] = model.predict_proba(X_test)[:, 1]

Engineering global semantic features...
Training Gradient Boosting Classifier...

Validation Accuracy: 68.42%
Validation ROC-AUC Score: 0.7835

Generating final veracity scores for 1342 test facts...


In [17]:

# STEP 4: SERIALIZATION & EXPORT TO RDF TURTLE FORMAT
# Generates GERBIL-compliant RDF Graph outputting veracity scores as xsd:double


# Define the predicate URI expected by GERBIL benchmark evaluation
HAS_TRUTH_VALUE = URIRef("http://swc2017.aksw.org/hasTruthValue")

print("Generating the result.ttl submission graph...")
output_graph = Graph()

# Iterate through test set predictions and build the RDF graph
for _, row in final_test.iterrows():
    fact_uri = URIRef(row["fact_uri"])

    # Format the predicted probability score to 4 decimal places
    score_str = f"{row['predicted_truth']:.4f}"

    # Cast score strictly as xsd:double datatype literal for GERBIL compatibility
    veracity_val = Literal(score_str, datatype=XSD.double)

    # Add triple: <Fact-URI> <http://swc2017.aksw.org/hasTruthValue> "0.xxxx"^^xsd:double
    output_graph.add((fact_uri, HAS_TRUTH_VALUE, veracity_val))

# Serialize the constructed RDF graph to Turtle (.ttl) format
output_filename = "result.ttl"
output_graph.serialize(destination=output_filename, format="turtle")

# Output export confirmation and absolute file path
print("--- DONE ---")
print(f"File exported successfully: {os.path.abspath(output_filename)}")
print("You can now upload this result.ttl to GERBIL.")

Generating the result.ttl submission graph...
--- DONE ---
File exported successfully: /content/result.ttl
You can now upload this result.ttl to GERBIL.
